# Modelos y resultados

Inicialmente, se presentarán algunos modelos clásicos aplicados a nuestro conjunto de datos.

## Modelos benchmark

Implementación de modelos benchmark con validarición cruzada estratificada y aleatoria.

In [ ]:
import pandas as pd
import numpy as np
import time
from sklearn.model_selection import StratifiedKFold, ShuffleSplit, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# El archivo 'poker-hand-training-true.data' posee 25010 filas y 'poker-hand-testing.data' posee un millón.
# Por esto, las variables en las que se guardan están invertidas.
# Si no se hiciera, se entrenaría con 25 mil instancias y se evaluaría con el millón.
train_path = "datosDePoquer/poker-hand-testing.data"
test_path = "datosDePoquer/poker-hand-training-true.data"

column_names = ["S1", "C1", "S2", "C2", "S3", "C3", "S4", "C4", "S5", "C5", "Clase"]

df_train = pd.read_csv(train_path, names=column_names)
df_test = pd.read_csv(test_path, names=column_names)

# Variables predictoras (X) y variable objetivo (y)
X_train = df_train.drop(columns=["Clase"])
y_train = df_train["Clase"]
X_test = df_test.drop(columns=["Clase"])
y_test = df_test["Clase"]

# Implementar modelos benchmark
models = {
    "KNN": KNeighborsClassifier(n_neighbors=3),
    "Naive Bayes": MultinomialNB(),
    "Regresión Logística": LogisticRegression(penalty=None, solver="newton-cg", random_state=42),
    "Ridge": LogisticRegression(penalty='l2', solver="newton-cg", random_state=42),
    "Lasso": LogisticRegression(penalty='l1', solver="liblinear", random_state=42),
    "Árbol de Decisión": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(n_jobs=4),
    "XGBoost": XGBClassifier(eval_metric='mlogloss'),
}

cv = StratifiedKFold(n_splits=3)
shuffle_split = ShuffleSplit(test_size=.4, train_size=.1, n_splits=3)

for name, model in models.items():
    start_time = time.time()
    scoresStratified = cross_val_score(model, X_train, y_train, cv=cv, scoring='accuracy')
    stratified_time = time.time() - start_time
    print(f'{name}: Precisión media en CV (StratifiedKFold) = {np.mean(scoresStratified):.4f}')
    print(f'Tiempo de cómputo (StratifiedKFold): {stratified_time:.4f} segundos')
    
    start_time = time.time()
    scoresShuffle = cross_val_score(model, X_train, y_train, cv=shuffle_split, scoring='accuracy')
    shuffle_time = time.time() - start_time
    print(f'{name}: Precisión media en CV (ShuffleSplit) = {np.mean(scoresShuffle):.4f}')
    print(f'Tiempo de cómputo (ShuffleSplit): {shuffle_time:.4f} segundos')


Explicación de XGBoost con Lime.

In [ ]:
import lime
import lime.lime_tabular

xgb_model = XGBClassifier(eval_metric='mlogloss')

# Explicación con LIME para XGBoost
xgb_model.fit(X_train, y_train)
explainer = lime.lime_tabular.LimeTabularExplainer(
    X_train.values, 
    mode='classification', 
    feature_names=X_train.columns.tolist(), 
    class_names=np.unique(y_train).astype(str), 
    discretize_continuous=True
)

sample = X_test.iloc[850].values
exp = explainer.explain_instance(sample, xgb_model.predict_proba, num_features=10)
exp.show_in_notebook()


Implementación de SVM.

In [ ]:
from sklearn.metrics import classification_report, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC

X_class_9 = X_train[y_train == 9]  # Filtramos todas las instancias de la clase 9
y_class_9 = y_train[y_train == 9]
X_class_8 = X_train[y_train == 8]  # Filtramos todas las instancias de la clase 8
y_class_8 = y_train[y_train == 8]

# Número total de instancias de las clases 9 y 8
num_class_9 = len(X_class_9)
num_class_8 = len(X_class_8)

# Filtrar el resto de las instancias de las otras clases
X_rest = X_train[y_train != 9]
y_rest = y_train[y_train != 9]
X_rest = X_rest[y_rest != 8]
y_rest = y_rest[y_rest != 8]

# Queremos un total de cien mil datos, y ya tenemos todas las instancias de la clases minoritarias
num_total = 100000
num_rest = num_total - num_class_9 - num_class_8

# Seleccionar estratificadamente las instancias de las otras clases
X_rest_sample, _, y_rest_sample, _ = train_test_split(X_rest, y_rest, train_size=num_rest, stratify=y_rest, random_state=42)

# Combinar las instancias seleccionadas de las clases minoritarias con las instancias restantes
X_final = np.concatenate((X_class_9, X_rest_sample), axis=0)
y_final = np.concatenate((y_class_9, y_rest_sample), axis=0)
X_final = np.concatenate((X_class_8, X_rest_sample), axis=0)
y_final = np.concatenate((y_class_8, y_rest_sample), axis=0)

# Dividir en entrenamiento (80%) y prueba (20%)
X_train, X_val, y_train, y_val = train_test_split(X_final, y_final, train_size=0.8, stratify=y_final, random_state=42)

# Implementación del modelo
model = SVC()
start_time = time.time()
model.fit(X_train, y_train)
y_pred = model.predict(X_val)
finish_time = time.time() - start_time
print("Precisión del modelo:", accuracy_score(y_val, y_pred))
print(f'Tiempo de cómputo: {finish_time:.4f} segundos')

## Balanceo de clases

Usando SMOTE, ADASYN y class_weight='balanced' de Scikit-learn con Máquina de soporte vectorial y Random Forest.

In [ ]:
import pandas as pd
import numpy as np
from imblearn.over_sampling import SMOTE, ADASYN
from collections import Counter
import time
from sklearn.model_selection import StratifiedKFold, ShuffleSplit, cross_val_score
from xgboost import XGBClassifier

train_path = "datosDePoquer/poker-hand-testing.data"
test_path = "datosDePoquer/poker-hand-training-true.data"

column_names = ["S1", "C1", "S2", "C2", "S3", "C3", "S4", "C4", "S5", "C5", "Clase"]

df_train = pd.read_csv(train_path, names=column_names)
df_test = pd.read_csv(test_path, names=column_names)

# Variables predictoras (X) y variable objetivo (y)
X_train = df_train.drop(columns=["Clase"])
y_train = df_train["Clase"]
X_test = df_test.drop(columns=["Clase"])
y_test = df_test["Clase"]

# Aplicar SMOTE
smote = SMOTE(sampling_strategy="auto", k_neighbors=1, random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)


print("Distribución original de clases:")
print(sorted(Counter(y_train).items()))

# Definir una estrategia de muestreo personalizada
# Solo equilibrar las clases 3-8 (ignorar las más raras/desequilibradas)
sampling_strategy = {
    3: 35000,  # Three of a kind
    4: 10000,   # Straight
    5: 5000,   # Flush
    6: 3500,   # Full house
    7: 800,   # Four of a kind
    8: 100    # Straight flush
    # Clase 9 (Royal flush) omitida intencionalmente debido a su extrema rareza
}

# Aplicar ADASYN
adasyn = ADASYN(sampling_strategy=sampling_strategy, random_state=42, n_neighbors=4)
X_train_adasyn, y_train_adasyn = adasyn.fit_resample(X_train, y_train)

print("\nDistribución de clases después de ADASYN:")
print(sorted(Counter(y_train_adasyn).items()))

cv = StratifiedKFold(n_splits=3)
shuffle_split = ShuffleSplit(test_size=.4, train_size=.1, n_splits=3)
xgb_model = XGBClassifier(eval_metric='mlogloss')

# XGBoost con SMOTE y ADASYN
def evaluar_modelo(X_train, y_train, metodo):
    start_time = time.time()
    scoresStratified = cross_val_score(xgb_model, X_train, y_train, cv=cv, scoring='accuracy')
    stratified_time = time.time() - start_time
    print(f'{metodo}: Precisión media en CV (StratifiedKFold) = {np.mean(scoresStratified):.4f}')
    print(f'Tiempo de cómputo (StratifiedKFold): {stratified_time:.4f} segundos')
    
    start_time = time.time()
    scoresShuffle = cross_val_score(xgb_model, X_train, y_train, cv=shuffle_split, scoring='accuracy')
    shuffle_time = time.time() - start_time
    print(f'{metodo}: Precisión media en CV (ShuffleSplit) = {np.mean(scoresShuffle):.4f}')
    print(f'Tiempo de cómputo (ShuffleSplit): {shuffle_time:.4f} segundos')
    
# Evaluar con SMOTE
evaluar_modelo(X_train_smote, y_train_smote, 'SMOTE')

# Evaluar con ADASYN
evaluar_modelo(X_train_adasyn, y_train_adasyn, 'ADASYN')

## Algoritmos de optimización

Algunos de los modelos usados anteriormente pueden ser modificados de manera que su rendimiento puede mejorar, en dependencia del dataset particular que se esté trabajando. En esta parte se implementan dichas modificaciones.

In [ ]:
import numpy as np
import time
from sklearn.model_selection import ShuffleSplit, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression

# Definir modelos KNN con estructuras KD-Tree y Ball Tree
models = {
    "KNN_kd": KNeighborsClassifier(n_neighbors=3, algorithm="kd_tree"),
    "KNN_ball": KNeighborsClassifier(n_neighbors=3, algorithm="ball_tree"),
    "Ridge": LogisticRegression(penalty='l2', solver="saga", random_state=42),
    "Lasso": LogisticRegression(penalty='l1', solver="saga", random_state=42),
}

# Parámetros de validación cruzada
shuffle_split = ShuffleSplit(test_size=0.4, train_size=0.1, n_splits=3)

# Evaluar modelos KNN de Scikit-learn
for name, model in models.items():
    start_time = time.time()
    scoresShuffle = cross_val_score(model, X_train, y_train, cv=shuffle_split, scoring='accuracy')
    shuffle_time = time.time() - start_time
    print(f'{name}: Precisión media en CV (ShuffleSplit) = {np.mean(scoresShuffle):.4f}')
    print(f'Tiempo de cómputo (ShuffleSplit): {shuffle_time:.4f} segundos')



Optimización de Naive Bayes

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import ShuffleSplit
import numpy as np
import time

# Definir modelo Naive Bayes
nb = MultinomialNB()

shuffle_split = ShuffleSplit(n_splits=3, test_size=0.4, train_size=0.1)

# Inicializar listas para almacenar métricas
nb_scores = []
nb_times = []

for train_idx, test_idx in shuffle_split.split(X_train, y_train):
    X_tr, X_te = X_train.iloc[train_idx], X_train.iloc[test_idx]
    y_tr, y_te = y_train.iloc[train_idx], y_train.iloc[test_idx]

    batch_size = 1000
    classes = np.unique(y_train)  # Asegurar que el modelo conoce todas las clases

    start_time = time.time()
    for i in range(0, len(X_tr), batch_size):
        X_batch = X_tr.iloc[i : i + batch_size]
        y_batch = y_tr.iloc[i : i + batch_size]
        nb.partial_fit(X_batch, y_batch, classes=classes)

    # Evaluar modelo
    score = nb.score(X_te, y_te)
    nb_scores.append(score)

    elapsed_time = time.time() - start_time
    nb_times.append(elapsed_time)

    print(f"Precisión en esta iteración: {score:.4f}")
    print(f"Tiempo de cómputo: {elapsed_time:.4f} segundos")

print(f"\nPrecisión media con ShuffleSplit: {np.mean(nb_scores):.4f}")
print(f"Tiempo de cómputo promedio: {np.mean(nb_times):.4f} segundos")



## Matrices de confusión

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_classification
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

X_train = df_train.drop(columns=["Clase"])
y_train = df_train["Clase"]
X_test = df_test.drop(columns=["Clase"])
y_test = df_test["Clase"]

X_class_9 = X_train[y_train == 9]  # Filtramos todas las instancias de la clase 9
y_class_9 = y_train[y_train == 9]
X_class_8 = X_train[y_train == 8]  # Filtramos todas las instancias de la clase 8
y_class_8 = y_train[y_train == 8]

# Número total de instancias de las clases 9 y 8
num_class_9 = len(X_class_9)
num_class_8 = len(X_class_8)

# Filtrar el resto de las instancias de las otras clases
X_rest = X_train[y_train != 9]
y_rest = y_train[y_train != 9]
X_rest = X_rest[y_rest != 8]
y_rest = y_rest[y_rest != 8]

# Queremos un total de ciento veinte mil datos, y ya tenemos todas las instancias de la clases minoritarias
num_total = 120000
num_rest = num_total - num_class_9 - num_class_8

# Seleccionar estratificadamente las instancias de las otras clases
X_rest_sample, _, y_rest_sample, _ = train_test_split(X_rest, y_rest, train_size=num_rest, stratify=y_rest, random_state=42)

# Combinar las instancias seleccionadas de las clases minoritarias con las instancias restantes
X_final = np.concatenate((X_class_9, X_rest_sample), axis=0)
y_final = np.concatenate((y_class_9, y_rest_sample), axis=0)
X_final = np.concatenate((X_class_8, X_rest_sample), axis=0)
y_final = np.concatenate((y_class_8, y_rest_sample), axis=0)

# Dividir en entrenamiento (80%) y prueba (20%)
X_train, X_val, y_train, y_val = train_test_split(X_final, y_final, train_size=0.8, stratify=y_final, random_state=42)

models = {
    "KNN": KNeighborsClassifier(n_neighbors=3),
    "Naive Bayes": MultinomialNB(),
    "Regresión Logística": LogisticRegression(penalty=None, solver="newton-cg", random_state=42),
    "Ridge": LogisticRegression(penalty='l2', solver="saga", random_state=42),
    "Lasso": LogisticRegression(penalty='l1', solver="saga", random_state=42),
    "Árbol de Decisión": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_jobs=4, random_state=42),
    "XGBoost": XGBClassifier(eval_metric='mlogloss', random_state=42),
    "SVM": SVC()
}

# Calcular y mostrar matriz de confusión para cada modelo
print("MATRICES DE CONFUSIÓN PARA CADA MODELO")
print("=====================================\n")

for name, model in models.items():
    print(f"\n{'-'*50}")
    print(f"MODELO: {name}")
    print(f"{'-'*50}")
    
    try:
        # Para Naive Bayes, necesitamos asegurarnos de que no hay valores negativos
        if name == "Naive Bayes":
            # Hacer los datos no negativos
            X_train_use = np.abs(X_train)
            X_test_use = np.abs(X_test)
        else:
            X_train_use = X_train
            X_test_use = X_test
            
        # Entrenar el modelo
        model.fit(X_train_use, y_train)
        
        # Hacer predicciones
        y_pred = model.predict(X_test_use)
        
        # Calcular matriz de confusión
        cm = confusion_matrix(y_test, y_pred)
        
        # Calcular precisión
        acc = accuracy_score(y_test, y_pred)
        
        # Mostrar resultados
        print(f"Precisión: {acc:.4f}")
        print("\nMatriz de Confusión:")
        
        # Obtener clases únicas
        classes = np.unique(y_test)
        
        # Encabezado de columnas
        header = "    " + " ".join([f"{cls:4d}" for cls in classes])
        print(header)
        
        # Imprimir matriz con etiquetas de filas
        for i, row in enumerate(cm):
            print(f"{classes[i]:2d} | {' '.join([f'{cell:4d}' for cell in row])}")
            
    except Exception as e:
        print(f"Error al procesar el modelo {name}: {str(e)}")

print("\n\nNOTA: Cada fila representa la clase real y cada columna la clase predicha.")
print("Por ejemplo, el valor en la fila 0, columna 1 indica cuántas instancias")
print("de la clase 0 fueron incorrectamente clasificadas como clase 1.")